In [2]:
print("Hello, World!")

Hello, World!


In [3]:
%pip install pypdf


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install langchain langchain_community


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF file
conhecimento = PyPDFLoader("conhecimento_merge.pdf").load()



/var/folders/_s/mzblbn6j57n03wbl2ylmsd4r0000gn/T/ipykernel_67451/3130107230.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/viniciuse/Meu Workspace/Projetos - Estudos/Engenharia de IA /Agente RAG - Milhas Aéreas /venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
len(conhecimento)

59

In [7]:
conhecimento

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-09-29T15:33:35+00:00', 'source': 'conhecimento_merge.pdf', 'total_pages': 59, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \nPrograma de Cartão da Edição MasterCard Standard \n \nInformações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços aos quais você terá acesso como \nportador de cartão preferencial. Esses benefícios e serviços estão em vigor para portadores do cartão \nMasterCard Standard qualificados à partir de 1 de novembro de 2023. Este Guia substitui qualquer guia \nou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem ser uma descrição completa de todos os termos, condições, limitações, exclusões ou outras \ndisposições de qualquer programa ou benefícios fornecidos por, para, ou emiti

In [8]:
print(conhecimento[10].page_content[:500])

5 
Versão: novembro 2023 
2021 
 
 
elétrico, Bomba d'água elétrica, Faca elétrica, Fatiador, Jarra elétrica, Sanduicheira, Waffer, Grill, 
Churrasqueira Elétrica, Tostador, Torradeira de pão, Balança digital, Máquina de fazer pão 
doméstica, Batedeira Elétrica,  Fritadeira elétrica e a gás,  Panela elétrica, Cozedor a vapor, Medidor de 
pressão, Massageador, Multiprocessador / Processador / Centrífuga de frutas,  Espremedor de frutas, 
Liquidificador / Mixer, Iogurteira, Sorveteira, Telefone co


In [9]:
%pip install "unstructured[pdf]"


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Load only PDF files from the directory
loader = DirectoryLoader(
    path="documentos",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)

docs = loader.load()

In [11]:
len(docs)

59

In [12]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

CAMINHO_PDFS = "documentos"

pdf_files = Path(CAMINHO_PDFS).glob("*.pdf")
pdfs = []

for file_path in pdf_files:
    nome_cartao = file_path.stem

    pages = PyPDFLoader(str(file_path)).load()
    full_content = "\n".join(page.page_content for page in pages)

    doc = Document(
        page_content=full_content,
        metadata={
            "source": str(file_path),
            "cartao": nome_cartao,
            "total_pages": len(pages)
        }
    )
    pdfs.append(doc)
    print(f"{file_path.name}: {len(pages)} páginas, {len(full_content)} caracteres")

len(pdfs)  # esperado: 3

GTB_platinum_Nov23.pdf: 29 páginas, 81469 caracteres
GTB_standard_Nov23.pdf: 6 páginas, 12752 caracteres
GTB_gold_Nov23.pdf: 24 páginas, 65166 caracteres


3

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer = tokenizer,
    chunk_size=1250, 
    chunk_overlap=150
)

chuncks = splitter.split_documents(pdfs)

len(chuncks)  # esperado: 20

36

In [15]:
%pip install langchain-ollama


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
%pip install faiss-cpu


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
from langchain_community.vectorstores import FAISS 
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(model="bge-m3")

vectorstore = FAISS.from_documents(
    chuncks, 
    embedding_model
)


In [18]:
#RETRIEVER 

retriever = vectorstore.as_retriever()

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [ 
        ("system", "Responda usando exclusivamente os conteúdo fornecido. Seja breve na resposta com no máximo 100 palavras.\n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)

from langchain_ollama import OllamaLLM

model_llm = OllamaLLM(
    model = "gemma3:1b",

)

from langchain_core.output_parsers import StrOutputParser

cadeia = prompt | model_llm | StrOutputParser()

In [ ]:
pergunta = "Como fazer um seguro viagem ?"

model_llm.invoke(pergunta)

'Fazer um seguro viagem é um passo importante para garantir tranquilidade e evitar imprevistos durante a sua viagem. Aqui está um guia completo sobre como fazer um seguro viagem, dividido em etapas:\n\n**1. Entender as Necessidades:**\n\nAntes de começar a procurar um seguro, avalie suas necessidades de saúde e viagem:\n\n*   **Localização da viagem:** Qual país você visitará? Alguns países exigem um seguro particular.\n*   **Distâncias da viagem:** Se você for viajar longas distâncias, um seguro de viagem abrangente pode ser crucial.\n*   **Tipo de atividades:** Vai praticar esportes, fazer trilha ou visitar sítios históricos? Alguns seguros oferecem cobertura extra.\n*   **Condições médicas:** Você tem alguma doença crônicamente? Ter um seguro com cobertura médica e hospitalar pode ser importante.\n*   **Despesas médicas:** É fundamental ter cobertura para despesas médicas como consultas, exames, hospitalizações e medicamentos.\n\n**2. Tipos de Segurados:**\n\n*   **Seguro Viage Bási

: 

In [ ]:

trechos = retriever.invoke("query")

contexto = "\n\n".join([trecho.page_content for trecho in trechos])

cadeia.invoke({
    "query": pergunta,
    "contexto": contexto
})